# Matrix Multiplication

Matrix multiplication is one of the most fundamental operations in machine
learning. Fully connected (linear) layers, the layers of an MLP, attention, and
many other building blocks are matrix products, and in most neural networks the
bulk of the computation time, both in training and at inference, is spent
multiplying matrices. This notebook covers:

1. Rows and columns of a matrix
2. The matrix product and its shapes
3. Batched matrix multiplication
4. The computational cost of a matrix product

In [ ]:
import time

import matplotlib.pyplot as plt
import torch

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False)


def check_shape(name, t, expected, symbols=""):
    # Print a tensor's shape (and optionally its symbolic shape), and assert it matches.
    print(f"{name:<10} {str(tuple(t.shape)):<16} {symbols}")
    assert tuple(t.shape) == tuple(expected), f"{name}: expected {tuple(expected)}, got {tuple(t.shape)}"

## 1. Rows and columns

A 2-D tensor of shape `(n, m)` is a matrix with
**$n$ rows** (the first dimension) and **$m$ columns** (the last dimension).
Each row is a vector of $m$ numbers. `M[i]` is row $i$, and `M[:, j]` is column $j$.

## 2. The matrix product

Multiplying an $(n, m)$ matrix by an $(m, p)$ matrix gives an
$(n, p)$ matrix. 

$$(n, m) \; @ \; (m, p) \;\to\; (n, p)$$

$$
\begin{array}{ccccc}
M_1 & & M_2 & & M_1 M_2 \\[4pt]
n\left\{\vphantom{\rule[-1.125em]{0pt}{2.25em}}\right.
\underbrace{\left[\phantom{\rule[-1.125em]{1.35em}{2.25em}}\right]}_{m}
& \times &
m\left\{\vphantom{\rule[-0.675em]{0pt}{1.35em}}\right.
\underbrace{\left[\phantom{\rule[-0.675em]{3.15em}{1.35em}}\right]}_{p}
& = &
n\left\{\vphantom{\rule[-1.125em]{0pt}{2.25em}}\right.
\underbrace{\left[\phantom{\rule[-1.125em]{3.15em}{2.25em}}\right]}_{p}
\end{array}
$$

In PyTorch this is `M1 @ M2` (same as `torch.matmul(M1, M2)`).

In [ ]:
n, m, p = 5, 3, 7
M1 = torch.randn(n, m)
M2 = torch.randn(m, p)

print("M1 =\n", M1, "\n")                                  # 5 rows, 3 columns
check_shape("M1", M1, (n, m), "(n, m)")
check_shape("M2", M2, (m, p), "(m, p)")

M12 = M1 @ M2
check_shape("M1 @ M2", M12, (n, p), "(n, p)")

## 3. Batched matrix multiplication

For tensors with more than two dimensions,
**only the last two dimensions are rows and columns**. All leading dimensions
are *batch dimensions*: the tensor is a stack of matrices.

$$(\dots, n, m) \; @ \; (\dots, m, p) \;\to\; (\dots, n, p)$$

Each batch element is multiplied independently, and leading dimensions broadcast:
a plain `(m, p)` matrix is reused for every batch element.

In [ ]:
M1_batched = torch.randn(2, n, m)
M2_batched = torch.randn(2, m, p)
M12_batched = M1_batched @ M2_batched
check_shape("M12_batched", M12_batched, (2, n, p), "(2, n, p)")

# Each batch element is an independent matrix multiply
for b in range(2):
    assert torch.allclose(M12_batched[b], M1_batched[b] @ M2_batched[b])

# Broadcasting: the single (m, p) matrix M2 is reused for every batch element
check_shape("M1_b @ M2", M1_batched @ M2, (2, n, p), "(2, n, p)")

## 4. Computational cost

**Cost of one entry.** Entry $(i, j)$ of the product is the dot product of
**row $i$** of $M_1$ with **column $j$** of $M_2$. Both have $m$ numbers:

$$(M_1 M_2)_{ij} = \sum_{k=1}^{m} (M_1)_{ik} \, (M_2)_{kj}$$

This takes $m$ multiplications and $m - 1$ additions, so about $m$
**multiply-adds**.

**Cost of the whole product.** The result has $n \cdot p$ entries, so

$$\underbrace{n \cdot p}_{\text{entries}} \times \underbrace{m}_{\text{per entry}} = n \cdot m \cdot p \text{ multiply-adds} \;\approx\; 2\,n\,m\,p \text{ floating-point operations (FLOPs)}$$

The computational complexity is $O(n \cdot m \cdot p)$. An easy way to remember
it: **multiply the three different sizes** in

$$(n, m) \; @ \; (m, p) \;\to\; (n, p),$$

counting the shared size $m$ only once. Besides the inputs, the result needs
memory for $n \cdot p$ numbers.